# MOSES: DNA Coloring Algorithm

Contributors: Dr. Jason Kahn (Brookhaven), Sarah Hong (Columbia)

This notebook shows the final workflow for how to create a lattice structure and paint it.

In [1]:
# Load this cell so that changes made in other files will be automatically reflected
%load_ext autoreload
%autoreload 2

In [2]:
# Run this cell to load all imports
import sys
sys.path.append('../')

import numpy as np
from PyQt6.QtWidgets import QApplication
from PyQt6.QtCore import QCoreApplication

from app.design.Designer import RunDesigner
from app.visualize.Visualizer import RunVisualizer
from algorithm.lattice.Lattice import Lattice
from algorithm.painting.Painter2 import Painter
from algorithm.symmetry.Rotation import NpRotationDict
from algorithm.painting.BindingFlexibility import BindingFlexibility

## Create the lattice
Run the below cell to open a GUI which allows you to quickly create arbitrary lattice structures. These lattices are encoded as numpy arrays and are saved to a (renamable) file called `lattice.npy` in the `notebooks/data` subdirectory. You can also use this feature to save and reload arbitrary lattice designs you've previously created.

In [3]:
%gui qt

if __name__ == '__main__':
    if not QCoreApplication.instance():
        app = QApplication(sys.argv)
    else:
        app = QCoreApplication.instance()
    
    designer = RunDesigner(app)
    input_lattice = designer.run()
    
    if input_lattice is not None:
        print('Lattice received.')
        # rename this to whatever you want to call your lattice
        np.save("data/lattice.npy", input_lattice) 
    else:
        print("No lattice received.")

No lattice received.


## Running the algorithm

### Load lattice from file
We can load in the lattice we created with `np.load(filename.npy)`. Since the lattice is just a numpy array, you can also create your own arbitrary lattices without the designer by creating arbitrary numpy arrays yourself.

Some examples to run:
- `double_oreo.npy`
- `perovskite.npy`

### Painting algorithm
All you need to do to run the algorithm is to create a Lattice instance, `compute_symmetries()`, create a Painter instance, and `paint_lattice()`.

The example workflow is detailed below.

In [7]:
np.save("data/mini_helix.npy", input_lattice)

In [32]:
# input_lattice = np.load('data/double_oreo.npy', allow_pickle=True)

# An example lattice structure you can load in
input_lattice = np.load('data/helix.npy')

# Initialize the data structures and compute all symmetries
lattice = Lattice(input_lattice)
lattice.compute_symmetries()

# Paint the lattice
# painter = Painter(lattice, verbose=True) # Set verbose=True to print debug
# # painter.str_paint_lattice()
# # painter.comp_paint_lattice()
# painter.paint_lattice()

In [43]:
%gui qt

if __name__ == '__main__':
    if not QCoreApplication.instance():
        app = QApplication(sys.argv)
    else:
        app = QCoreApplication.instance()
    
    visualizeWindow = RunVisualizer(lattice=lattice, voxels=lattice.voxels, app=app) # insert bf_lattice here if you want
    # visualizeWindow = RunVisualizer(lattice=lattice, voxels=str_voxels, app=app) # insert bf_lattice here if you want

In [46]:
voxel_7 = lattice.get_voxel(7)
symdict_7 = lattice.symmetry_df.symdict(4)

for partner_id, symlist in symdict_7.items():
    print(f"(4, {partner_id}): {symlist}")

(4, 4): ['translation', '180° X-axis + 270° Z-axis', '180° Y-axis + 90° Z-axis']
(4, 14): ['180° Y-axis', '270° Z-axis', '180° Z-axis + 180° X-axis']
(4, 26): ['180° Z-axis', '180° X-axis + 90° Z-axis', '180° Y-axis + 180° X-axis', '180° Y-axis + 270° Z-axis']
(4, 34): ['180° X-axis', '90° Z-axis', '180° Z-axis + 180° Y-axis']


In [54]:
print(f"MinDesign shape: {lattice.MinDesign.shape}")
print(f"Full surroundings shape: {lattice.Surroundings.FullSurroundings.shape}")
print(f"Tile repeats: {lattice.Surroundings.tile_repeats}")

MinDesign shape: (4, 3, 3)
Full surroundings shape: (12, 15, 15)
Tile repeats: [3, 5, 5]


In [7]:
lattice.MinDesign

array([[[0, 0, 0],
        [0, 1, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 1],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 1]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 1, 0]]])

In [15]:
voxel_7 = lattice.get_voxel(7)
voxel_7_surr = lattice.Surroundings.voxel_surroundings(voxel_7, verbose=True)

print("\nSurroundings for voxel 7")
voxel_7_surr

Halfway points for each direction: ((4, 6, 6))
Slicing full_surr @ ((4, 8, 7))

Surroundings for voxel 7


array([[[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]],

       [[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]],

       [[0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0]],

       [[0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0]],

       [[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]]])

In [16]:
print("Voxel 5 surroundings\n")

voxel_5 = lattice.get_voxel(5)
voxel_5_surr = lattice.Surroundings.voxel_surroundings(voxel_5)

voxel_5_surr

Voxel 5 surroundings



array([[[0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0]],

       [[0, 1, 0, 0, 1],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 1, 0, 0, 1],
        [0, 0, 0, 0, 0]],

       [[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 1, 0, 0, 1],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]],

       [[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]],

       [[0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0]]])

In [23]:
# test out whether the double rotation itself is wrong or not

rot = surr_rotater.all_rotations['180° X-axis']
arr = np.array([[[1, 2],
                 [3, 4]],
                [[5, 6],
                 [7, 8]]])
print(f"Original arr\n{arr}")
rot_arr = rot(arr)
print(f"180x rotation\n{rot_arr}")

rot2 = surr_rotater.all_rotations['180° X-axis + 90° Z-axis']
rot_arr2 = rot2(arr)
print(f"180x + 90z rotation\n{rot_arr2}")

Original arr
[[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]]
180x rotation
[[[7 8]
  [5 6]]

 [[3 4]
  [1 2]]]
180x + 90z rotation
[[[8 6]
  [7 5]]

 [[4 2]
  [3 1]]]


In [17]:
surr_rotater = NpRotationDict()

# rot = surr_rotater.all_rotations['180° X-axis + 270° Z-axis']
rot = surr_rotater.all_rotations['180° X-axis + 90° Z-axis']

rot_surr_7 = rot(voxel_7_surr)
rot_surr_7

array([[[0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0]],

       [[0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0]],

       [[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]],

       [[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]],

       [[0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0]]])

In [18]:
print(f"Rotated voxel 7 surr = voxel 5 surr?{np.array_equal(rot_surr_7, voxel_5_surr)}")

Rotated voxel 7 surr = voxel 5 surr?False


In [25]:
s_voxels = painter.mesovoxel.structural_voxels
c_voxels = painter.mesovoxel.complementary_voxels

min_voxels = s_voxels | c_voxels
print(f"min voxel set: {min_voxels}")

min_voxels = [lattice.get_voxel(mv) for mv in min_voxels]
str_voxels = [lattice.get_voxel(sv) for sv in s_voxels]

min voxel set: {0, 1, 2, 4, 5, 8, 27, 28, 29}


## Optional: Binding Flexibility
In this section, we introduce an optional "binding flexibility" parameter, which modifies the final painting result to be stricter or looser. This is motivated by the observation that the geometry / symmetries of specific structures may perform better under different constraints, and we offer the user the option to make it more/less strict.

### Background
**Binding flexibility 1:** Reduces specificity (less colors)
- BF 1 turns all bonds to voxels in the same automorphism equivalence class are repainted to be the same color. 
- In other words, we repaint all bonds between all voxels with some symmetry between them to be the same color.

**Binding flexibility 2:** No change
- BF 2 is the default result from the Painter.paint() algorithm. It generally leads to the best results, so most systems are painted with this.

**Binding flexibility 3:** Adds specificity (more colors)
- Introducing a maximum cutoff ratio (= structural bonds / total bonds) per voxel.
- For all voxels in the lattice with a cutoff ratio higher than the maximum, we repaint a NEW color ontop of a structural bond on that voxel to lower the ratio to satisfy the constraint.
- This ratio is motivated by the observation that having more structural bonds on the same voxel makes it more likely to erroneously bind in ways we don't want.

### Usage
To repaint the lattice with a specific binding flexibility, create a new `BindingFlexibility` instance specified with a given (painted) lattice object. Then run either `binding_flexibility1()` or `binding_flexibility3()` and assign it to a new variable to create a NEW lattice instance which can be visualized in the app. An example usage can be run below.

In [7]:
bf = BindingFlexibility(lattice)

bf1_lattice = bf.binding_flexibility_1()
bf3_lattice = bf.binding_flexibility_3(max_cutoff_ratio=1/6)

## Visualize the painted lattice

Run the below cell to view the painted lattice as a result from the algorithm.

To view a specific binding flexibility in the app, just supply the given binding flexibility lattice as the argument to `RunVisualizer`.

In [7]:
%gui qt

if __name__ == '__main__':
    if not QCoreApplication.instance():
        app = QApplication(sys.argv)
    else:
        app = QCoreApplication.instance()
    
    visualizeWindow = RunVisualizer(lattice=lattice2, voxels=lattice2.voxels.values(), app=app) # insert bf_lattice here if you want
    # visualizeWindow = RunVisualizer(lattice=lattice, voxels=str_voxels, app=app) # insert bf_lattice here if you want

In [18]:
for voxel in lattice.voxels:
    print(f"Voxel{voxel.id} is {voxel.type}")

Voxel0 is structural
Voxel1 is complementary
Voxel2 is complementary
Voxel3 is structural
Voxel4 is structural
Voxel5 is complementary
Voxel6 is complementary
Voxel7 is structural
Voxel8 is complementary
Voxel9 is structural
Voxel10 is structural
Voxel11 is complementary
Voxel12 is complementary
Voxel13 is structural
Voxel14 is structural
Voxel15 is complementary


## Minimal origami set

You can run `lattice.final_df()` to get an interpretable, final dataframe which contains all voxel/bond information.

In [ ]:
lattice.unique_origami()

In [17]:
final_df = lattice.final_df()
final_df

Voxel                      Bond Colors               
     ID Material Coordinates          +x -x +y -y +z -z
0     0        1   (0, 1, 1)           1  1  1  1  1  1
1     1        0   (1, 1, 1)          -1 -1  2  2  2  2
2     2        0   (0, 0, 1)           2  2 -1 -1  2  2
3     3        2   (1, 0, 1)          -2 -2 -2 -2  3  3
4     4        0   (0, 1, 0)           2  2  2  2 -1 -1
5     5        2   (1, 1, 0)          -2 -2  3  3 -2 -2
6     6        2   (0, 0, 0)           3  3 -2 -2 -2 -2
7     7        3   (1, 0, 0)          -3 -3 -3 -3 -3 -3

In [7]:
from scipy.spatial.transform import Rotation
import numpy as np

# new type of voxel surroundings - as dictionary
surr = {
    (1, 0, 0): 1,
    (0, 1, 0): 0,
    (0, 0, 1): 1,
    (1, 1, 1): 0
}

rotation = Rotation.from_euler('z', 360, degrees=True) # get this from the dict later

# convert coords and materials to their own np.arrays
surr_keys = np.array(list(surr.keys()))
surr_values = list(surr.values())

rot_surr_keys = rotation.apply(surr_keys)
rot_surr_keys = np.round(rot_surr_keys).astype(int)
rot_surr = {tuple(key): value for key, value in zip(rot_surr_keys, surr_values)}

if set(surr.keys()) == set(rot_surr.keys()):
    all_match = all(surr[key] == rot_surr[key] for key in surr.keys())

else:
    all_match = False

print(f"all materials match after rotation?: {all_match}")

rot_surr


all materials match after rotation?: True


{(np.int64(1), np.int64(0), np.int64(0)): 1,
 (np.int64(0), np.int64(1), np.int64(0)): 0,
 (np.int64(0), np.int64(0), np.int64(1)): 1,
 (np.int64(1), np.int64(1), np.int64(1)): 0}

In [8]:
# Define the maximum dimension
max_dim = 3  # Example value

# Generate 1D array of coordinates for each axis
coords_range = np.linspace(-max_dim, max_dim, num=2*max_dim + 1)

# Generate the 3D grid of coordinates
x, y, z = np.meshgrid(coords_range, coords_range, coords_range, indexing='ij')

# Flatten the grids and combine them into a list of 3D coordinates
coords = np.array([x.flatten(), y.flatten(), z.flatten()]).T

print("All possible coordinates within the cube:")
print(coords)

All possible coordinates within the cube:
[[-3. -3. -3.]
 [-3. -3. -2.]
 [-3. -3. -1.]
 ...
 [ 3.  3.  1.]
 [ 3.  3.  2.]
 [ 3.  3.  3.]]


In [5]:
from algorithm.lattice.Lattice2 import Lattice2
from algorithm.symmetry.Surroundings2 import Surroundings2

# An example lattice structure you can load in
input_lattice = np.load('data/helix.npy')

# Initialize the data structures and compute all symmetries
lattice2 = Lattice2(input_lattice)
lattice2.compute_symmetries()

lattice2.symmetry_df.symmetry_df

,translation,90° X-axis,180° X-axis,270° X-axis,90° Y-axis,180° Y-axis,270° Y-axis,90° Z-axis,180° Z-axis,270° Z-axis,...,90° X-axis + 90° Z-axis,90° Y-axis + 180° X-axis,90° Y-axis + 180° Z-axis,90° Y-axis + 270° X-axis,90° Y-axis + 270° Z-axis,90° Y-axis + 90° X-axis,90° Y-axis + 90° Z-axis,90° Z-axis + 180° X-axis,90° Z-axis + 270° X-axis,90° Z-axis + 270° Y-axis
"(4, 25)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(2, 8)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(7, 31)",False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(13, 34)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(15, 30)",False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"(14, 26)",False,False,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
"(9, 33)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(11, 20)",False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(27, 34)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [6]:
painter2 = Painter(lattice2, verbose=True) # Set verbose=True to print debug
# painter.str_paint_lattice()
# painter.comp_paint_lattice()
painter2.paint_lattice()

===== STARTING STR_PAINT_LATTICE =====

--- PAINT S_BOND (1) --- 
voxel_0 ((1, 0, 0)) <---> voxel_1 ((-1, 0, 0))
self-sym paint(voxel_0)
    map_paint(parent_0 --> child_0, sym=translation, with_negation=False)
    map_paint(parent_0 --> child_0, sym=180° Y-axis + 270° Z-axis, with_negation=False)
    map_paint(parent_0 --> child_0, sym=270° Z-axis + 180° X-axis, with_negation=False)
self-sym paint(voxel_1)
    map_paint(parent_1 --> child_1, sym=translation, with_negation=False)

--- PAINT S_BOND (2) --- 
voxel_0 ((-1, 0, 0)) <---> voxel_2 ((1, 0, 0))
self-sym paint(voxel_0)
    map_paint(parent_0 --> child_0, sym=translation, with_negation=False)
    map_paint(parent_0 --> child_0, sym=180° Y-axis + 270° Z-axis, with_negation=False)
    map_paint(parent_0 --> child_0, sym=270° Z-axis + 180° X-axis, with_negation=False)
self-sym paint(voxel_2)
    map_paint(parent_2 --> child_2, sym=translation, with_negation=False)

--- PAINT S_BOND (3) --- 
voxel_1 ((1, 0, 0)) <---> voxel_2 ((-1, 0,

In [50]:
painter2.mesovoxel.complementary_voxels

set()

In [34]:
df2 = lattice2.symmetry_df.symmetry_df
df2

,translation,90° X-axis,180° X-axis,270° X-axis,90° Y-axis,180° Y-axis,270° Y-axis,90° Z-axis,180° Z-axis,270° Z-axis,...,270° Z-axis + 180° X-axis,270° Z-axis + 180° Y-axis,270° Z-axis + 90° X-axis,270° Z-axis + 90° Y-axis,90° Y-axis + 180° Z-axis,90° Y-axis + 90° X-axis,90° Z-axis + 180° X-axis,90° Z-axis + 180° Y-axis,90° Z-axis + 90° X-axis,90° Z-axis + 90° Y-axis
"(4, 25)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(2, 8)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(7, 31)",False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(13, 34)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(15, 30)",False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"(14, 26)",False,False,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
"(9, 33)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(11, 20)",False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(27, 34)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [35]:
# original sym_df
df1 = lattice.symmetry_df.symmetry_df
df1

,translation,90° X-axis,180° X-axis,270° X-axis,90° Y-axis,180° Y-axis,270° Y-axis,90° Z-axis,180° Z-axis,270° Z-axis,...,270° Z-axis + 180° X-axis,270° Z-axis + 180° Y-axis,270° Z-axis + 90° X-axis,270° Z-axis + 90° Y-axis,90° Y-axis + 180° Z-axis,90° Y-axis + 90° X-axis,90° Z-axis + 180° X-axis,90° Z-axis + 180° Y-axis,90° Z-axis + 90° X-axis,90° Z-axis + 90° Y-axis
"(4, 25)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(2, 8)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(7, 31)",False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(13, 34)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(15, 30)",False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"(14, 26)",False,False,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
"(9, 33)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(11, 20)",False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
"(27, 34)",False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
